In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
name = "aveera"
RESULTS_DIR = f"/data/{name}/results"
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

INCLUDE_RANDOM = False                              
METHOD_ORDER = ["GradCAM", "LIME", "SHAP"] + (["Random"] if INCLUDE_RANDOM else [])

RUNS = {
    "original": ("rise_original_summary.csv", "original tie-breaking"),
    "custom":   ("rise_custom_summary.csv",   "seeded tie-breaking"),
}

def bar_chart(summary, value_col, title, ylabel, save_path):
    pivot = (summary[summary["method"].isin(METHOD_ORDER)]
             .pivot(index="method", columns="model", values=value_col)
             .reindex(METHOD_ORDER))

    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    
    colors = ["#006D77", "#83C5BE", "#4D908E", "#264653", "#A8DADC"]
    
    fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)
    
    pivot.plot(
        kind="bar", 
        ax=ax, 
        width=0.75, 
        color=colors[:len(pivot.columns)], 
        edgecolor="white", 
        linewidth=1.2,
        zorder=3
    )

    ax.set_title(title, fontsize=13, fontweight="bold", pad=20, color="#1A1A1A", loc="left")
    ax.set_ylabel(ylabel, fontsize=11, fontweight="medium", labelpad=12, color="#333333")
    ax.set_xlabel("Method", fontsize=11, fontweight="medium", labelpad=12, color="#333333")
    
    ax.tick_params(colors="#4A4A4A", labelsize=10)
    plt.xticks(rotation=0)
    
    ax.grid(visible=True, axis="y", linestyle="--", linewidth=0.7, color="#E0E0E0", zorder=0)
    ax.grid(visible=False, axis="x")
    
    for spine in ["top", "right", "bottom", "left"]:
        ax.spines[spine].set_visible(False)

    ax.legend(
        title="Model", 
        title_fontsize=10,
        fontsize=9,
        loc="upper left", 
        bbox_to_anchor=(1.02, 1), 
        frameon=True,
        facecolor="#F8F9FA",
        edgecolor="#E0E0E0",
        shadow=False
    )
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

for tag, (csv_name, label) in RUNS.items():
    path = os.path.join(RESULTS_DIR, csv_name)
    if not os.path.exists(path):
        print(f"Skipping {tag}: {path} not found")
        continue
    summary = pd.read_csv(path)

    print(f"{label.title()}")
    bar_chart(summary, "del_auc",
                f"RISE Deletion Faithfulness by Model and Method",
                "Mean Deletion AUC\n(lower = more faithful)",
                os.path.join(FIGURES_DIR, f"rise_{tag}_deletion_bars.png"))

    bar_chart(summary, "ins_auc",
                f"RISE Insertion Faithfulness by Model and Method",
                "Mean Insertion AUC\n(higher = more faithful)",
                os.path.join(FIGURES_DIR, f"rise_{tag}_insertion_bars.png"))